### Setup

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (0/0):
Data count before: 480608
Data count after: 480608
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480608
Num of distinct users: 2103
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480608
interaction data count after merging: 478564
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358129 (74.83%
valid: 46916 (9.8%)
test: 73519 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555886
1    0.444114
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585155
1    0.414845
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358129
  Num of positive interactions: 159050 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 159050], edge_label=[159050])
Edge Index: tensor([[   0,    0,    0,  ..., 2093, 2093, 2093],
        [1102, 1186,  670,  ..., 2835,  742, 2969]])


#### Prepare prediction pool for inference/testing

In [8]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2095
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2095(users) * 500(items) = 1047500


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1047495,2094,7957,0,"[5403, 102, 5844, 14185, 8714]",63,955,"[8, 15, 0, 0, 0, 0, 0, 0]"
1047496,2094,1611,0,"[13198, 4752, 15371, 5573, 8531]",63,2910,"[4, 5, 0, 0, 0, 0, 0, 0]"
1047497,2094,4715,0,"[2653, 2800, 5578, 9181, 11244]",63,1393,"[7, 18, 0, 0, 0, 0, 0, 0]"
1047498,2094,1924,0,"[3069, 7855, 11716, 8412, 5873]",62,2293,"[5, 0, 0, 0, 0, 0, 0, 0]"
1047499,2094,1590,0,"[6359, 5740, 13573, 3459, 14058]",63,2954,"[1, 2, 8, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [9]:
# TODO: determine which Dataset to use
from common.datasets import UserItemPairDataset
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

train_dataset = UserItemPairDataset(encoded_train_df)
valid_dataset = UserItemPairDataset(encoded_valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 358129
valid data count: 46916
test data count: 1047500


### Configure Model (LightningModule)

In [11]:
from models.gcn_cf_rec import GCNRecCF

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3

model = GCNRecCF(
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,  # shape [2, num_edges]
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
)


### Configure Trainer and Experiment

In [15]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "gcn-bce-exp"
RUN_NAME = "gcn-baseline-test4"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(EXPERIMENT_NAME, RUN_NAME, PATIENCE)

In [16]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [17]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/experiments/test_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type            | Params | Mode 
------------------------------------------------------
0 | gcn_model | GraphConvModule | 815 K  | train
------------------------------------------------------
815 K     Trainable params
0         Non-trainable params
815 K     Total params
3.261     Total estimated model params size (MB)
43        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 24, global step 8750: 'val_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_f1 did not improve in the last 5 records. Best score: 0.523. Signaling Trainer to stop.
Epoch 25, global step 9100: 'val_f1' was not in top 1


🏃 View run gcn-baseline-test4 at: http://140.112.106.216:3683/#/experiments/3/runs/c45a79f6368848d9a21076ec4574b0b3
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


### Inference

In [19]:
# NOTE: the inference model MUST be the same as the training model
best_model_path = "test_checkpoints/gcn-bce-exp-gcn-baseline-test4-best-checkpoint-epoch=20-val_f1=0.52.ckpt"

model = GCNRecCF.load_from_checkpoint(
    checkpoint_path=best_model_path,
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
)


In [20]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6927303075790405     │
│          test_f1          │    0.1262761801481247     │
│         test_loss         │     4.234642505645752     │
│         test_prec         │    0.0688372328877449     │
│         test_rec          │    0.7626151442527771     │
└───────────────────────────┴───────────────────────────┘

🏃 View run gcn-baseline-test4 at: http://140.112.106.216:3683/#/experiments/3/runs/c45a79f6368848d9a21076ec4574b0b3
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/3


[{'test_loss': 4.234642505645752,
  'test_acc': 0.6927303075790405,
  'test_prec': 0.0688372328877449,
  'test_rec': 0.7626151442527771,
  'test_f1': 0.1262761801481247}]

- gcn-bce-exp-gcn-baseline-test2-best-checkpoint-epoch=04-val_f1=0.45.ckpt
<details>
K=500

- [{'test_loss': 364.225341796875,
- 'test_acc': 0.7884315252304077,
- 'test_prec': 0.05633168667554855,
- 'test_rec': 0.39781633019447327,
- 'test_f1': 0.09868881106376648}]

K=250

- [{'test_loss': 372.6000061035156,
- 'test_acc': 0.7848190665245056,
- 'test_prec': 0.07433757930994034,
- 'test_rec': 0.3977948725223541,
- 'test_f1': 0.1252661496400833}]
</details>

- gcn-bce-exp-gcn-baseline-test3-best-checkpoint-epoch=00-val_f1=0.57.ckpt
<details>
K=500

- [{'test_loss': 12384.9208984375,
- 'test_acc': 0.5445107221603394,
- 'test_prec': 0.049183208495378494,
- 'test_rec': 0.7988131046295166,
- 'test_f1': 0.09266123175621033}]
</details>

In [21]:
model.test_results

{'user': tensor([   0,    0,    0,  ..., 2094, 2094, 2094]),
 'item': tensor([7977, 4839,  979,  ..., 4715, 1924, 1590]),
 'score': tensor([-1.1870,  0.1562,  4.9048,  ..., -0.6301, -2.8669, -6.2242]),
 'label': tensor([1., 0., 1.,  ..., 0., 0., 0.]),
 'metric': {'test_loss': 4.234642505645752,
  'test_acc': 0.6927303102625298,
  'test_prec': 0.06883723408033526,
  'test_rec': 0.7626151677104167,
  'test_f1': 0.12627618538314744},
 'user_emb': tensor([[-2.6230e+00, -1.5077e+00, -1.7998e+00,  ..., -4.7933e-01,
          -1.0685e+00, -1.4195e+00],
         [-1.2587e+00, -7.3385e-01, -8.3407e-01,  ..., -2.5605e-01,
          -4.9733e-01, -6.4812e-01],
         [-2.2625e-01, -1.3243e-01, -1.4843e-01,  ..., -4.7627e-02,
          -9.0951e-02, -1.1651e-01],
         ...,
         [-7.3151e+00, -4.1963e+00, -4.9766e+00,  ..., -1.3496e+00,
          -2.9514e+00, -3.8983e+00],
         [-1.3313e+01, -7.5893e+00, -8.9268e+00,  ..., -2.5832e+00,
          -5.2219e+00, -6.8372e+00],
         [ 9.9

In [23]:
eval_df = evaluator.prepare_evaluation_data(model.test_results)
eval_df

,user,rec_items,gt_items
0,0,"[2419, 2090, 281, 316, 4932, 4142, 5946, 6006,...","[7977, 979, 97, 2419, 2090]"
1,1,"[2390, 7031, 968, 2321, 3165, 7005, 1364, 3111...","[3392, 5801, 6570, 8192]"
2,2,"[4507, 1746, 2426, 2392, 7455, 6678, 5860, 710...","[7978, 5768]"
3,3,"[260, 49, 7031, 969, 3303, 739, 231, 3246, 754...","[969, 3246, 3303, 2066, 6880, 7908, 7979, 2932..."
4,4,"[7060, 2534, 6856, 1790, 4893, 5, 4970, 7805, ...","[1492, 1139, 4085, 8160, 6253, 1380, 4395, 8212]"
...,...,...,...
2090,2090,"[1865, 2534, 1014, 1002, 5566, 1275, 621, 1539...","[1596, 1790, 5627, 977, 6914, 8311, 1587, 823,..."
2091,2091,"[6799, 6487, 4643, 1865, 3303, 5706, 6170, 515...","[8140, 8017, 8126, 5941, 8214, 8075]"
2092,2092,"[870, 4932, 7546, 6170, 946, 97, 1344, 3289, 8...","[3289, 4932, 645, 1496, 1889, 1326]"
2093,2093,"[2330, 4643, 6123, 3068, 739, 637, 5706, 6170,...","[974, 7805, 6123, 3491, 2189, 2392, 1010, 1007..."


In [24]:
eval_score_df = evaluator.evaluate(eval_df, K=5)
eval_score_df = evaluator.evaluate(eval_score_df, K=10)
eval_score_df = evaluator.evaluate(eval_score_df, K=20)
eval_score_df.describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000
mean,1047.000000,0.256587,0.051196,0.121718,0.304634,0.098137,0.118329,0.349623,0.190899,0.116229
std,604.918727,0.351721,0.098290,0.184139,0.323943,0.147863,0.150606,0.279171,0.214116,0.121453
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,523.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1047.000000,0.000000,0.000000,0.000000,0.301030,0.037037,0.100000,0.350282,0.133333,0.100000
75%,1570.500000,0.501266,0.066667,0.200000,0.543771,0.142857,0.200000,0.549952,0.285714,0.200000
max,2094.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.800000,1.000000,1.000000,0.750000
